# CHIP PRS using hapmam3 European ancestry-specific bld-ldak reference panel

In [16]:
###############################################################################
#             0) Clean workspace and load necessary libraries
###############################################################################
rm(list = ls())         # clear everything
gc()                    # force garbage collection

# Libraries
library(data.table)     # for fread, data.table manipulations
library(survival)       # for coxph
library(fst)            # fast I/O to disk
library(future.apply)   # parallel 'future_lapply'
library(progressr)      # nice progress bars for parallel tasks

###############################################################################
##

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,1969655,105.2,3402975,181.8,3402975,181.8
Vcells,10741437,82.0,102073328,778.8,127419893,972.2


In [3]:
##
setwd("/medpop/esp2/mesbah/projects/Meta_GWAS/MetaGWAS_N900k/PheWAS/")
#########

In [20]:
###############################################################################
#             1) Read and merge your UKB datasets + PheCODE outcomes
#                (Replace these paths and merges with your own)
###############################################################################

cat("1) Loading/merging large data...\n")
#### CHIP data in UKB
# # CHIP VAR
# ukbbch_var <- fread("/Volumes/medpop_esp2/mesbah/datasets/CHIP/UKBB/ukb450_2023/ch_var_all_ukb200k_n_ukb250k.7Mar2023.csv.gz", header=T)
# head(ukbbch_var)
# sort(table(ukbbch_var$Gene), decreasing = T)
# 
# sort(table(ukbbch_var$Protein_Change[ukbbch_var$Gene%in% c("IDH1", "IDH2")]), decreasing = T)

# CHIP: /medpop/esp2/mesbah/tools/CHIP_metaAnalysis//04.SNP_heritability/QuickPRS
ukb200k_chip <- fread("/medpop/esp2/mesbah/datasets/CHIP/UKBB/ukb450_2023/CH_phenoCovar.ukb200k_N193342.28cols.03_05_2024.tsv.gz", header=T)

ukb250k_chip <- fread("/medpop/esp2/mesbah/datasets/CHIP/UKBB/ukb450_2023/CH_phenoCovar.ukb250k_N243350.28cols.03_05_2024.tsv.gz", header=T)
table(ukb200k_chip$FID %in% ukb250k_chip$FID)
ukb450k_chip <- as.data.frame(rbind(ukb200k_chip, ukb250k_chip)); rm(ukb200k_chip, ukb250k_chip)

cat("unique sample size", length(unique(ukb450k_chip$FID)))

head(ukb450k_chip)  

table(ukb450k_chip$Batch, exclude = NULL)

## UKB baseline info 
## UKB Baseline
d_base <- fread("/medpop/esp2/mesbah/projects/gxe/UKB_baseline/UKB500k_baseline_info_recoded_participant.tsv.gz")
# "ever_smoked.p20160_i0",         
# "Smoking_status.p20116_i0",
table(d_base$p20116_i0,  d_base$p20160_i0, exclude = NULL)
## Smoking status
ukb450k_chip$smking_status <- factor(ifelse(ukb450k_chip$FID %in% d_base$eid[d_base$p20160_i0=="Yes"], 
                                            1, 
                                            ifelse(ukb450k_chip$FID %in% d_base$eid[d_base$p20160_i0=="No"],
                                                   0, NA)))
table(ukb450k_chip$smking_status, exclude = NULL)

rm(d_base)

##### regorup
table(ukb450k_chip$knn, exclude = NULL)
table(ukb450k_chip$Batch, exclude = NULL)
table(ukb450k_chip$Genetic_Sex, exclude = NULL)
table(ukb450k_chip$GenoBatch, exclude = NULL)
table(ukb450k_chip$smking_status, exclude = NULL)

ukb450k_chip$Genetic_Ancestry_group <- case_when(
    ukb450k_chip$knn == "EUR" ~ "EUR", 
    ukb450k_chip$knn == "AFR" ~ "AFR", 
    ukb450k_chip$knn == "SAS" ~ "SAS", 
TRUE ~ "AMR_or_EAS")

names(ukb450k_chip)
ls()

table(ukb450k_chip$knn, ukb450k_chip$Genetic_Ancestry_group, exclude = NULL)

1) Loading/merging large data...



 FALSE 
193342 

unique sample size 436692

,FID,IID,hasCH,hasCHvaf05,hasCHvaf10,hasDNMT3A,hasTET2,hasASXL1,hasDTA,hasSF,...,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,Batch,knn
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>
1,1000015,1000015,0,0,0,0,0,0,0,0,...,1.224590,4.77684,2.318020,2.445040,-3.067290,-1.08792,0.550964,1.201910,ukb200k,EUR
2,1000042,1000042,1,1,1,NA,1,NA,1,NA,...,-1.486040,-3.42686,-5.359130,-0.146506,2.788340,-4.25511,-6.282060,-0.862379,ukb200k,EUR
3,1000059,1000059,0,0,0,0,0,0,0,0,...,0.274298,-1.35773,1.917950,-1.627780,-0.191782,-1.28348,2.726250,1.320520,ukb200k,EUR
4,1000061,1000061,0,0,0,0,0,0,0,0,...,-0.557439,1.85166,-0.349985,-3.238940,-2.291060,-1.76164,-0.691142,1.272970,ukb200k,EUR
5,1000113,1000113,0,0,0,0,0,0,0,0,...,-2.160480,1.20825,-3.988450,-0.436352,-0.639480,-4.37838,2.542470,2.549570,ukb200k,EUR
6,1000151,1000151,0,0,0,0,0,0,0,0,...,-3.346580,1.94703,8.926610,0.703991,-1.088360,1.60221,-0.529712,-3.535610,ukb200k,EUR



ukb200k ukb250k 
 193342  243350 

...


     Axiom UKBiLEVEAX 
    393260      43432 


     0      1   <NA> 
174364 260136   2192 

[1] "FID"                    "IID"                    "hasCH"                 
 [4] "hasCHvaf05"             "hasCHvaf10"             "hasDNMT3A"             
 [7] "hasTET2"                "hasASXL1"               "hasDTA"                
[10] "hasSF"                  "hasDDR"                 "Ethnic_Background"     
[13] "sqrtAge_at_recruitment" "GenoBatch"              "Age_at_recruitment"    
[16] "Genetic_Sex"            "PC1"                    "PC2"                   
[19] "PC3"                    "PC4"                    "PC5"                   
[22] "PC6"                    "PC7"                    "PC8"                   
[25] "PC9"                    "PC10"                   "Batch"                 
[28] "knn"                    "smking_status"          "Genetic_Ancestry_group"

[1] "ukb450k_chip"                     "ukb450k_chip_eur"                
[3] "ukb450k_chip_eur.multi_n_eur_prs"

     
         AFR AMR_or_EAS    EUR    SAS
  AFR   8558          0      0      0
  AMR      0       2210      0      0
  EAS      0       2398      0      0
  EUR      0          0 413857      0
  SAS      0          0      0   9669

In [21]:

######
## PRS with EUR Reference Panel
load("/medpop/esp2/mesbah/projects/Meta_GWAS/MetaGWAS_N900k/sumHer/prs/prs.eur.ukb450k.rda")

ukb450k_chip_eur <- merge(ukb450k_chip, dat_eur, 
                          by.x="FID", by.y="ID2"); rm(ukb450k_chip, dat_eur)
head(ukb450k_chip_eur)
names(ukb450k_chip_eur)
# multianc gwas prs
# ukb450k_chip_eur_multianc <- ukb450k_chip_eur[,c(1,2,13:21,27:29,35,38,44,50,32,41,47)]
# cat("unique sample size", length(unique(ukb450k_chip_eur_multianc$FID)))
# names(ukb450k_chip_eur_multianc)
### EUR-ancestry gwas PRS
# ukb450k_chip_eur_eurprs <- ukb450k_chip_eur[,c(1,2,13:21,27:29,34,37,43,49,31,40,46)]
# cat("unique sample size", length(unique(ukb450k_chip_eur_eurprs$FID)))
# names(ukb450k_chip_eur_eurprs)
# 
ukb450k_chip_eur.multi_n_eur_prs <- ukb450k_chip_eur[,c(1,2,13:21,27,29,30,35,36,
                                                        38,39,44,45,50,51,32,33,
                                                        41,42,47,48)]
cat("unique sample size", length(unique(ukb450k_chip_eur.multi_n_eur_prs$FID)))
names(ukb450k_chip_eur.multi_n_eur_prs)

,FID,IID,hasCH,hasCHvaf05,hasCHvaf10,hasDNMT3A,hasTET2,hasASXL1,hasDTA,hasSF,...,DDR_MultiANC,DNMT3A_AFR,DNMT3A_EUR,DNMT3A_MultiANC,SF_AFR,SF_EUR,SF_MultiANC,TET2_AFR,TET2_EUR,TET2_MultiANC
,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,...,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1000015,1000015,0,0,0,0,0,0,0,0,...,0.011565,-0.086815,-0.070390,0.009825,0.003253,-0.045317,0.005940,-0.087192,0.007390,0.005348
2,1000023,1000023,0,0,0,0,0,0,0,0,...,-0.022768,0.054543,-0.041316,0.011650,0.042174,0.002742,-0.018633,-0.028722,-0.031789,0.004244
3,1000037,1000037,0,0,0,0,0,0,0,0,...,0.037276,0.048966,0.049025,0.070851,-0.053061,0.006364,0.002662,-0.117677,0.039820,-0.048886
4,1000042,1000042,1,1,1,NA,1,NA,1,NA,...,-0.016930,-0.023074,0.025860,0.025954,0.043006,0.008090,-0.010029,0.082055,0.074607,0.079228
5,1000059,1000059,0,0,0,0,0,0,0,0,...,-0.000429,-0.069071,0.145097,0.069124,0.051019,0.056864,0.051727,-0.100094,0.063040,-0.031384
6,1000061,1000061,0,0,0,0,0,0,0,0,...,0.013645,0.057205,0.073656,0.084201,-0.042158,0.037000,0.027767,-0.028710,-0.000818,0.001138


[1] "FID"                    "IID"                    "hasCH"                 
 [4] "hasCHvaf05"             "hasCHvaf10"             "hasDNMT3A"             
 [7] "hasTET2"                "hasASXL1"               "hasDTA"                
[10] "hasSF"                  "hasDDR"                 "Ethnic_Background"     
[13] "sqrtAge_at_recruitment" "GenoBatch"              "Age_at_recruitment"    
[16] "Genetic_Sex"            "PC1"                    "PC2"                   
[19] "PC3"                    "PC4"                    "PC5"                   
[22] "PC6"                    "PC7"                    "PC8"                   
[25] "PC9"                    "PC10"                   "Batch"                 
[28] "knn"                    "smking_status"          "Genetic_Ancestry_group"
[31] "ASXL1_AFR"              "ASXL1_EUR"              "ASXL1_MultiANC"        
[34] "CH_AFR"                 "CH_EUR"                 "CH_MultiANC"           
[37] "CHvaf10_AFR"            "CHvaf10_EUR"            "CHvaf10_MultiANC"      
[40] "DDR_AFR"                "DDR_EUR"                "DDR_MultiANC"          
[43] "DNMT3A_AFR"             "DNMT3A_EUR"             "DNMT3A_MultiANC"       
[46] "SF_AFR"                 "SF_EUR"                 "SF_MultiANC"           
[49] "TET2_AFR"               "TET2_EUR"               "TET2_MultiANC"

unique sample size 435810

[1] "FID"                    "IID"                    "sqrtAge_at_recruitment"
 [4] "GenoBatch"              "Age_at_recruitment"     "Genetic_Sex"           
 [7] "PC1"                    "PC2"                    "PC3"                   
[10] "PC4"                    "PC5"                    "Batch"                 
[13] "smking_status"          "Genetic_Ancestry_group" "CH_EUR"                
[16] "CH_MultiANC"            "CHvaf10_EUR"            "CHvaf10_MultiANC"      
[19] "DNMT3A_EUR"             "DNMT3A_MultiANC"        "TET2_EUR"              
[22] "TET2_MultiANC"          "ASXL1_EUR"              "ASXL1_MultiANC"        
[25] "DDR_EUR"                "DDR_MultiANC"           "SF_EUR"                
[28] "SF_MultiANC"

## CHIP PRS: European ancestry-specific AOU+TOPMED+MGB GWAS weight 

In [23]:
# Phecode categories
phenCategory <- c("CirculatoryRespiratory", 
                  "HematologicNeoplasmInfectious",
                  "DermDigestGU",
                  "MentalNeuroSensorySymptoms", 
                  "InjuriesPoisoningMSK",
                  "PregnancyCongenitalEndocrine")

In [ ]:

for(k in 1:length(phenCategory) ){
  gc()
  cat(k,"\n")
    cat(phenCategory[k],"\n")

  dat <- fread(paste("/medpop/esp2/mzekavat/UKBB/PhenoFiles/PheCODEs_new/2021-01-08_ukb_phecode_", phenCategory[k], "_March2020fu.csv", sep=""), sep=",")
  # dat <- fread(paste("../../../PheWAS/UKBB_MGBB_MVP_BioVU/PheCODEs_new/2021-01-08_ukb_phecode_", phenCategory[k], "_March2020fu.csv.gz", sep=""), sep=",")
  
  dat <- dat %>% select(1, contains("_INCID"), contains("_DAYS"))
  
  # Multi-ancestry and EUR GWAS PRS with EUR ref
  ukb_prs_multianc <- merge(ukb450k_chip_eur.multi_n_eur_prs, 
                            dat,
                            by.x="FID", 
                            by.y="f.eid")
  rm(dat)
  gc()
  
   
  cat("Data dimension:", dim(ukb_prs_multianc), "\n")
  
  
  ###############################################################################
  #             2) Identify your outcomes and exposures
  ###############################################################################
  # E.g. you found outcomes by searching for "_INCID" columns, then removing suffix:
  outcomes <- gsub(
    pattern     = "_INCID", 
    replacement = "", 
    x           = names(ukb_prs_multianc)[grepl("INCID", names(ukb_prs_multianc), ignore.case = TRUE)]
  )
  cat("Number of outcomes found:", length(outcomes), "\n")
  
  # 7 exposures from your code:
  # Define exposures
  # exposures <- c("CH_MultiANC","CHvaf10_MultiANC","DNMT3A_MultiANC",
    #             "TET2_MultiANC","ASXL1_MultiANC",
     #            "DDR_MultiANC","SF_MultiANC")
    
  exposures <- c('CH_EUR', 'CH_MultiANC', 'CHvaf10_EUR', 'CHvaf10_MultiANC', 
                 'DNMT3A_EUR', 'DNMT3A_MultiANC', 'TET2_EUR', 
                 'TET2_MultiANC', 'ASXL1_EUR', 'ASXL1_MultiANC', 
                 'DDR_EUR', 'DDR_MultiANC', 'SF_EUR', 'SF_MultiANC')
  
  
  cat("Number of exposures:", length(exposures), "\n")
  
  cat("Total # of Cox fits to run =", length(outcomes)*length(exposures), "\n")
  
    ###############################################################################
  #             3) Write to .fst (fast on-disk format), then remove from memory
  ###############################################################################
  # This avoids sending a giant data object to each worker.
  
    library(fst)
  
  
    cat("Saving ukb_prs_multianc.fst to disk...\n")
  
    write_fst(ukb_prs_multianc, paste0(phenCategory[k],".ukb_prs_multianc.fst"))
  
    rm(ukb_prs_multianc)
  
    gc()
  
  ###############################################################################
  #             4) Chunk the 224 outcomes so we don't do them all at once
  ###############################################################################
  # We'll process them in sets of e.g. 20 outcomes per chunk.
  
    chunk_size <- 10
  
    outcome_indices <- seq(1, length(outcomes), by = chunk_size)
  
  ###############################################################################
  #             5) Define a function to run Cox models for a single outcome
  #                on your 7 exposures
  ###############################################################################
  run_coxph_for_one_outcome <- function(outcome_prefix, exposures, progressor = NULL) {
    # 1) Read from disk inside the worker
    local_data <- as.data.table(read_fst(paste0(phenCategory[k],".ukb_prs_multianc.fst")) )
    
    # 2) Subset rows, e.g., exclude first 30 days
    outcome_incid <- paste0(outcome_prefix, "_INCID")
    outcome_days  <- paste0(outcome_prefix, "_DAYS")
    local_data <- local_data[get(outcome_days) > 30]
    
    # 3) Drop unused factor levels
    local_data <- droplevels(local_data)
    
    # 4) Check if any factor has <2 levels
   # factor_vars <- c("knn", "Genetic_Sex", "smking_status", "Batch", "GenoBatch")
     factor_vars <- c("Genetic_Ancestry_group", "Genetic_Sex")
      
    for (fv in factor_vars) {
      if (fv %in% names(local_data)) {
        if (length(unique(local_data[[fv]])) < 2) {
          # Skip this outcome to avoid the contrasts error
          return(NULL)
        }
      }
    }
    
    # 5) Count events
    disease_cases <- sum(local_data[[outcome_incid]] == 1, na.rm = TRUE)
    total_n       <- nrow(local_data)
    if (disease_cases == 0 || total_n == 0) {
      return(NULL)
    }
    
    # 6) Loop over exposures, fit cox, store results
    results_for_this_outcome <- list()
    for (expo in exposures) {
      if (!is.null(progressor)) {
        progressor(sprintf("Outcome=%s, Exposure=%s", outcome_prefix, expo))
      }
      
      fml <- as.formula(
        paste0(
          "Surv(", outcome_days, ", ", outcome_incid, ") ~ scale(", expo, 
          ") + Genetic_Ancestry_group + Genetic_Sex + Age_at_recruitment + smking_status + ",
          "Batch + GenoBatch + sqrtAge_at_recruitment + PC1 + PC2 + PC3 + PC4 + PC5"
        )
      )
      
      model <- coxph(fml, data = local_data)
      coef_data <- summary(model)$coefficients[1, c(1, 3, 4, 5)]  # Beta, SE, Z, P
      
      results_for_this_outcome[[length(results_for_this_outcome) + 1]] <- data.table(
        Outcome       = outcome_prefix,
        Exposure      = expo,
        Beta          = coef_data[1],
        SE            = coef_data[2],
        Z             = coef_data[3],
        P             = coef_data[4],
        Disease_Cases = disease_cases,
        N             = total_n
      )
    }
    
    return(rbindlist(results_for_this_outcome))
  }
  
  ###############################################################################
  #             6) Main loop over outcome CHUNKS (parallel within each chunk)
  ###############################################################################
  # We'll accumulate results in a list. Each chunk returns a data.table of results.
  all_chunks_results <- list()
  chunk_counter <- 1
  
  # Set up parallel plan (keep it modest if memory is a concern)
  library(future)
  plan(multisession, workers = 5)
  
  cat("\nStarting chunked analysis...\n")
  
  for (start_idx in outcome_indices) {
    end_idx <- min(start_idx + chunk_size - 1, length(outcomes))
    chunk_outcomes <- outcomes[start_idx:end_idx]
    
    cat(sprintf("Processing chunk #%d: outcomes [%d..%d] (size %d)\n",
                chunk_counter, start_idx, end_idx, length(chunk_outcomes)))
    
    # Show progress bar for this chunk
    with_progress({
      p <- progressor(steps = length(chunk_outcomes))
      
      # future_lapply() for parallel runs of each outcome in the chunk
      # chunk_results_list <- future_lapply(
      #   X   = chunk_outcomes,
      #   FUN = function(one_outcome) {
      #     run_coxph_for_one_outcome(
      #       outcome_prefix = one_outcome,
      #       exposures      = exposures,
      #       progressor     = p
      #     )
      #   }
      # )
      # 
      chunk_results_list <- future_lapply(
        X   = chunk_outcomes,
        FUN = function(one_outcome) {
          run_coxph_for_one_outcome(
            outcome_prefix = one_outcome,
            exposures      = exposures,
            progressor     = p
          )
        },
        future.seed = TRUE  # <--- Important for reproducible, parallel-safe RNG
      )
      
      # Combine chunk results
      chunk_results <- rbindlist(chunk_results_list, fill=TRUE)
      all_chunks_results[[chunk_counter]] <- chunk_results
    })
    
    chunk_counter <- chunk_counter + 1
  }
  
  # Combine *all* chunks
  final_results <- rbindlist(all_chunks_results, fill = TRUE)
  
  
  ###############################################################################
  #             7) Save final results
  ###############################################################################
  cat("\nWriting final combined results...\n")
  
  output_dir <- paste0(phenCategory[k],"_UKB450k_chip_prs")
  
  dir.create(output_dir, showWarnings = FALSE)
  
  out_file <- paste0(output_dir,"/",phenCategory[k],".UKB_Cox_Results.csv")
  
  fwrite(final_results, out_file)
  
  cat("All done! Results in ", out_file,"\n") 

}

############################### END #########################  
  
  